# Parkinson’s Voice Classification: Leakage-Aware Subject Evaluation

This notebook demonstrates the methodology for subject-level evaluation on the UCI Parkinsons dataset.

- **Problem**: 195 recordings from 32 subjects (repeated measurements per person).
- **Leakage**: A naive record-level split leaks subject identity, producing optimistic accuracy (~92%).
- **Solution**: Subject-level nested cross-validation (4 outer x 3 inner folds) with L2 Logistic Regression and median aggregation.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from parkinson_voice.data import load_data, ORIGINAL_FEATURES, TARGET_COLUMN, SUBJECT_COLUMN
from parkinson_voice.features import MODEL_FEATURES, REDUNDANT_FEATURES
from parkinson_voice.audit import run_naive_split_audit
from parkinson_voice.model_selection import nested_subject_cross_fitted
from parkinson_voice.predict import load_bundle, predict_subject_records

## 1. Load & Inspect Dataset
Check total recordings vs unique subjects.

In [ ]:
data_path = 'data/parkinsons.csv' if os.path.exists('data/parkinsons.csv') else '../data/parkinsons.csv'
frame = load_data(data_path)

print(f'Total recordings: {len(frame)}')
print(f'Total unique subjects: {frame[SUBJECT_COLUMN].nunique()}')
print(f'Subject class distribution:\n{frame.groupby(SUBJECT_COLUMN)[TARGET_COLUMN].first().value_counts()}')

## 2. Leakage Demonstration (Naive Split vs Subject Split)
Demonstrate how random row-level splitting causes data leakage.

In [ ]:
audit_result = run_naive_split_audit(data_path=data_path)
print('Naive record-level split test accuracy:', f"{audit_result['accuracy']:.3f}")
print(f"Overlapping subjects in test: {audit_result['overlapping_test_subjects']} / {audit_result['test_subjects']}")
print('=> All test subjects overlap with train subjects! This causes optimistic bias.')

## 3. Nested Subject Cross-Validation (4 Outer x 3 Inner)
Evaluate model generalization with zero leakage between train and test subjects.

In [ ]:
fold_metrics, cross_fitted, selections = nested_subject_cross_fitted(
    frame,
    outer_splits=4,
    inner_splits=3,
    random_state=42,
)

print('Outer fold performance:')
print(fold_metrics[['Outer fold', 'C', 'class_weight', 'Balanced Accuracy', 'ROC-AUC', 'F1-macro']])
print(f"\nPooled Balanced Accuracy: {fold_metrics['Balanced Accuracy'].mean():.4f}")
print(f"Pooled ROC-AUC: {fold_metrics['ROC-AUC'].mean():.4f}")

## 4. Inference with Deployment Model
Load the saved model and predict a sample subject.

In [ ]:
artifact_path = 'artifacts/model.joblib' if os.path.exists('artifacts/model.joblib') else '../artifacts/model.joblib'
bundle = load_bundle(artifact_path)

sample_sub_id = frame[SUBJECT_COLUMN].iloc[0]
sample_recordings = frame[frame[SUBJECT_COLUMN] == sample_sub_id].drop(
    columns=['status', 'name', 'subject_id'], errors='ignore'
).to_dict('records')

result = predict_subject_records(sample_sub_id, sample_recordings, bundle)
print('Prediction result for', sample_sub_id, ':')
print(json.dumps(result, indent=2, ensure_ascii=False))